# 6 - Multivariable Regression

Fit `life_expectancy ~ income + education + poverty + race + age + log_pop` and compare to the single-variable results.

**Question this notebook is built to answer:** Once we control for income, education, and poverty, does the apparent racial gap in life expectancy persist or shrink? And which structural variable carries the most independent explanatory weight?

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
df = pd.read_csv('data/merged_df.csv', dtype={'fips': str}).dropna(
    subset=['life_expectancy','median_hh_income','pct_bachelors_plus',
            'pct_poverty','pct_nhwhite','median_age','log_population']
)
print('n =', len(df))

### Three nested models

In [ ]:
socioecon = ['median_hh_income','pct_bachelors_plus','pct_poverty']
demo      = ['pct_nhwhite','median_age','log_population']
all_p     = socioecon + demo
y = df['life_expectancy']

m_socio = sm.OLS(y, sm.add_constant(df[socioecon])).fit()
m_demo  = sm.OLS(y, sm.add_constant(df[demo])).fit()
m_full  = sm.OLS(y, sm.add_constant(df[all_p])).fit()

summary = pd.DataFrame({
    'model':  ['Socioeconomic only (3)', 'Demographic only (3)', 'Full (6)'],
    'R2':     [round(m_socio.rsquared, 3),    round(m_demo.rsquared, 3),    round(m_full.rsquared, 3)],
    'adj_R2': [round(m_socio.rsquared_adj, 3), round(m_demo.rsquared_adj, 3), round(m_full.rsquared_adj, 3)],
    'AIC':    [round(m_socio.aic, 1),         round(m_demo.aic, 1),         round(m_full.aic, 1)],
})
summary

### Full model coefficients

In [ ]:
print(m_full.summary())

**Reading the table:** The slopes are in years-of-life-expectancy per unit of X.

- `median_hh_income` is in dollars, so its slope will be tiny per dollar - divide by 1000   to read it as 'years per $1,000 of income'.
- `pct_bachelors_plus` is in percentage points, so the slope is years per 1pp increase.
- `pct_nhwhite` slope, after socioeconomic controls, will likely shrink dramatically vs. its   single-variable counterpart in notebook 5. That's the headline finding.

### Single vs. multivariable significance - the key plot

In [ ]:
single_p = {p: sm.OLS(y, sm.add_constant(df[p])).fit().pvalues[p] for p in all_p}
multi_p  = {p: m_full.pvalues[p] for p in all_p}

fig, ax = plt.subplots(figsize=(10, 5))
xs = np.arange(len(all_p)); w = 0.38
ax.bar(xs - w/2, [-np.log10(max(single_p[p], 1e-300)) for p in all_p], w,
       label='Single-variable regression', color='#7fb069')
ax.bar(xs + w/2, [-np.log10(max(multi_p[p], 1e-300))  for p in all_p], w,
       label='Multivariable regression',   color='#d8453b')
ax.axhline(-np.log10(0.05), color='gray', linestyle='--', linewidth=1, label='p = 0.05')
ax.set_xticks(xs); ax.set_xticklabels(all_p, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('-log10(p-value) - higher = more significant')
ax.set_title('How significance shifts when other variables are controlled for')
ax.legend()
plt.tight_layout(); plt.show()

### Coefficient comparison: single-variable vs. multivariable slopes

In [ ]:
single_b = {p: sm.OLS(y, sm.add_constant(df[p])).fit().params[p] for p in all_p}
multi_b  = {p: m_full.params[p] for p in all_p}

comparison = pd.DataFrame({
    'predictor':       all_p,
    'single_var_coef': [round(single_b[p], 4) for p in all_p],
    'multivar_coef':   [round(multi_b[p],  4) for p in all_p],
    'shrinkage':       [f'{(multi_b[p]/single_b[p]*100):.0f}%' if single_b[p] != 0 else 'n/a' for p in all_p],
})
comparison

**The 'shrinkage' column is the punchline.** A coefficient that goes from significant in single-variable to small/insignificant in multivariable means its apparent effect was a stand-in for something else (typically: income or education).

Watch `pct_nhwhite` in particular. If its single-variable slope is ~+0.05 (i.e., counties that are 1pp more white have ~0.05 years higher life expectancy), and its multivariable slope drops to ~+0.01 or even goes negative, that's the story: the racial gap in county life expectancy is mostly explained by the income/education gap. (Which doesn't make it less of a problem - it relocates the explanation.)

### R-squared stepwise

In [ ]:
m_inc = sm.OLS(y, sm.add_constant(df['median_hh_income'])).fit()

fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Income\n(only)', 'Demographic\n(3)', 'Socioeconomic\n(3)', 'Full\n(6)']
vals = [m_inc.rsquared, m_demo.rsquared, m_socio.rsquared, m_full.rsquared]
colors = ['#3b7dd8','#d8453b','#7fb069','#5e3bd8']
bars = ax.bar(labels, vals, color=colors, edgecolor='white')
ax.set_ylabel('R-squared (variance in life expectancy explained)')
ax.set_ylim(0, 1)
ax.set_title('Variance in county life expectancy explained, by model')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.show()

---

## After you run this notebook

Open `memo.md` and fill in the placeholders with the actual numbers from your output:
- Full-model R-squared
- Coefficient on `median_hh_income` (per $1,000) and `pct_bachelors_plus`
- The single-vs-multivar shrinkage for `pct_nhwhite`
- Names of the highest- and lowest-life-expectancy counties from notebook 2

Then write up the story angle that fits what you found.

## Caveats
- Cross-sectional, observational data. Not causal.
- ACS estimates are noisier in small-population counties; consider weighting by population   if the standard errors look implausibly tight.
- County is a coarse geography. The same county can contain neighborhoods with 10+ year   life-expectancy gaps. Census tract analysis would be more revealing.
- Some counties drop out due to FIPS-code changes (esp. Connecticut 2022 reorg) or CHR   suppression of small-population estimates.